In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import os
import warnings
warnings.filterwarnings('ignore')

RAW_DIR = "../data/raw"
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Carga de Datos y Consolidación de Train + Test
Para calcular el `lag_52` en el conjunto de prueba (Nov 2012), necesitamos que mire hacia atrás a los datos de Nov 2011, los cuales viven en el archivo de entrenamiento. Por ende, unimos ambos temporalmente.

In [ ]:
# Cargar datos base
train = pd.read_csv(os.path.join(RAW_DIR, "train.csv"))
test = pd.read_csv(os.path.join(RAW_DIR, "test.csv"))
stores = pd.read_csv(os.path.join(RAW_DIR, "stores.csv"))
features = pd.read_csv(os.path.join(RAW_DIR, "features.csv"))

# Identificar a qué set pertenece cada fila
train['is_test'] = 0
test['is_test'] = 1

# El test no tiene Weekly_Sales, la creamos vacía para hacer concat
test['Weekly_Sales'] = np.nan

# Unimos Train y Test
combined = pd.concat([train, test], ignore_index=True)

# Unimos tiendas y features externas
combined['Date'] = pd.to_datetime(combined['Date'])
features['Date'] = pd.to_datetime(features['Date'])

df = combined.merge(stores, on='Store', how='left')
df = df.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')

# ORDEN CRUCIAL PARA LOS LAGS
df = df.sort_values(by=['Store', 'Dept', 'Date']).reset_index(drop=True)
print(f"Total combinados: {len(df)} filas (Train: {len(train)}, Test: {len(test)})")

## 2. Feature Engineering
Replicamos exactamente la lógica del Notebook 02, aplicándola al dataset unificado.

In [ ]:
# 1. Tratamiento nulos en MarkDowns
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
for col in markdown_cols:
    df[f'Has_{col}'] = np.where(df[col].notnull(), 1, 0)
    df[col] = df[col].fillna(0)

# 2. Variables Temporales
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsHoliday'] = df['IsHoliday'].astype(int)
df['Type'] = df['Type'].astype('category').cat.codes

# 3. Lags Estacionales (Seguros => horizontes >= 39)
lag_periods = [52, 53, 56]
for lag in lag_periods:
    df[f'Weekly_Sales_lag_{lag}'] = df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(lag)

print("Features temporales y lags generados.")

## 3. Separación Final (Train vs Test) y Selección de Features

In [ ]:
# Volvemos a dividir en base a la flag
df_train = df[df['is_test'] == 0].copy()
df_test = df[df['is_test'] == 1].copy()

# Limpieza de columnas
drop_cols = ['Date', 'Weekly_Sales', 'is_test', 'Id']
features_cols = [c for c in df_train.columns if c not in drop_cols]

X_train = df_train[features_cols]
y_train = df_train['Weekly_Sales']
X_test = df_test[features_cols]

print(f"Entrenaremos sobre {len(X_train)} filas y predeciremos {len(X_test)} de test.")

## 4. Entrenamiento de LightGBM y XGBoost con Sample Weights y Categorical Features
Aquí aplicamos el mismo esquema de modelado, pero entrenando con el 100% de la historia (Train).  
Utilizamos los pesos `train_weights` para penalizar x5 el error en semanas de holidays, y le indicamos explícitamente a LightGBM las variables categóricas.

In [ ]:
# Crear vector de pesos exacto al de la métrica oficial (5x para holidays)
train_weights = np.where(df_train['IsHoliday'] == 1, 5, 1)

# Definición explícita de features categóricas para evitar que los algoritmos de árbol 
# realicen particiones tratándolas erróneamente como variables continuas ordinales.
cat_features = ['Store', 'Dept', 'Type', 'Week']

print("Entrenando LightGBM con pesos y categorías explícitas...")
# Pasamos las categorías al dataset de LightGBM
lgb_train = lgb.Dataset(X_train, label=y_train, weight=train_weights, categorical_feature=cat_features)
lgb_params = {
    'objective': 'regression_l1', 
    'metric': 'mae',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'feature_fraction': 0.8,
    'n_jobs': -1,
    'seed': 42
}
# Usamos unas 900 iteraciones como estándar prudente sin val-set
final_lgb = lgb.train(lgb_params, lgb_train, num_boost_round=900)
lgb_preds = final_lgb.predict(X_test)

print("Entrenando XGBoost con pesos...")
# XGBoost más moderno maneja categorías si se lo configuramos o confía en la estructura
dtrain = xgb.DMatrix(X_train, label=y_train, weight=train_weights)
dtest = xgb.DMatrix(X_test)
xgb_params = {
    'objective': 'reg:absoluteerror',
    'eval_metric': 'mae',
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.8,
    'n_jobs': -1,
    'seed': 42
}
final_xgb = xgb.train(xgb_params, dtrain, num_boost_round=600)
xgb_preds = final_xgb.predict(dtest)

print("¡Modelos entrenados!")

## 5. Exportación de Modelos a Producción
Por buenas prácticas (MLOps), guardamos los modelos inmediatamente después de ser entrenados.

In [ ]:
import joblib

MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

# 1. Guardar modelo LightGBM (formato nativo .txt para lectura rápida)
lgb_model_path = os.path.join(MODELS_DIR, "lightgbm_final.txt")
final_lgb.save_model(lgb_model_path)

# 2. Guardar modelo XGBoost (formato nativo .json)
xgb_model_path = os.path.join(MODELS_DIR, "xgboost_final.json")
final_xgb.save_model(xgb_model_path)

print(f"Modelos serializados exitosamente en la carpeta: {MODELS_DIR}")

## 6. Ensamble, Post-procesamiento y Archivo Submission

In [ ]:
# 1. Ensamble Optimizado (Validado con WMAE)
best_w = 0.60
final_preds = (best_w * lgb_preds) + ((1.0 - best_w) * xgb_preds)

# 2. Post-procesamiento: Cortar ventas negativas
final_preds = np.clip(final_preds, 0, None)

# 3. Formato Kaggle: Id (Store_Dept_Date) y Weekly_Sales
df_test['Id'] = df_test['Store'].astype(str) + '_' + df_test['Dept'].astype(str) + '_' + df_test['Date'].dt.strftime('%Y-%m-%d')
df_test['Weekly_Sales'] = final_preds

# 4. Guardar archivo CSV
submission = df_test[['Id', 'Weekly_Sales']]
sub_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(sub_path, index=False)

print(f"Archivo generado en: {sub_path}")
submission.head()